In [2]:
from langchain.tools import tool
import requests
from dotenv import load_dotenv
import os
from tavily import TavilyClient
from rich import print
from bs4 import BeautifulSoup
from readability import Document
import trafilatura
import re   ## regular expression (regex)

load_dotenv()

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def search_web(query :str) -> str :    ##  -> this sign show that this fuction return string
        """ Search the web for recent and relaible information on a topic. Return title URL and"""

        results = tavily.search(query=query, max_result=5)

        out = []

        for r in results['results'] :
                out.append(
                        f"Title : {r['title']}\nURL : {r['url']}\nSnippet: {r['content'][:300]}\n"
                )


        return "\n----\n".join(out)

In [3]:
search_web("Which new movie is releasing in india this week?")


"Title : Upcoming Movies 2026 — New Releases This Week | District\nURL : https://www.district.in/movies/upcoming-movies\nSnippet: January 14, 2027##### Children of Blood and Bone\n\nJanuary 15, 2027##### The Pride of Bharat: Chhatrapati Shivaji Maharaj\n\nJanuary 21, 2027##### Love & War\n\nJanuary 21, 2027##### The Rescue\n\nJanuary 29, 2027\n\nFebruary 2027\n\n##### Remain\n\nFebruary 5, 2027##### Tera Sai\n\nFebruary 5, 2027##### Ice Age: \n\n----\nTitle : New Hindi Movies | List of Latest Bollywood Movie Releases 2026 - Gadgets 360\nURL : https://www.gadgets360.com/entertainment/new-hindi-movies\nSnippet: Image 129: Grand Theft Auto 6 Will Get an Extended Look on Netflix on August 27, Rockstar Announces Netflix Reportedly in Talks to Bring Rival Streaming Services to Its Platform\n   Image 130: Grand Theft Auto 6 Will Get an Extended Look on Netflix on August 27, Rockstar Announces OTT Releases This W\n\n----\nTitle : Upcoming Movies 2026 | List of Movies Releasing This Week in Mum

In [4]:
def scrape_url(url:str)  -> str:
        """
        Scrape and extract clean readable content from a URL.
        Uses multiple extraction strategies for better reliability.
        """

        headers ={
                "User-Agent":(
                        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/124.0 Safari/537.36"
                ),
                "Accept-Language": "en-US,en:q=0.9",
                "Referer":"https://www.google.com",
        }

        try:
                # fetch page
                response= requests.get(
                        url,
                        headers=headers,
                        timeout=15
                )

                response.raise_for_status()

                html=response.text

                # Strategy 1 -> trafilatura (Besst for articale/blogs)

                extracted = trafilatura.extract(
                        html,
                        include_comments = False,
                        include_tables = False,
                )

                if extracted and len(extracted.strip()) > 200 :
                        cleaned = re.sub(r'\s+',' ', extracted)
                        return cleaned[:5000]
                
                # Strategy 2 -> readability

                doc = Document(html)
                clean_html = doc.summary()

                soup = BeautifulSoup (clean_html, " html_parser")

                for tag in soup([
                        "script",
                        "style",
                        "nav",
                        "footer",
                        "header",
                        "aside",
                        "form"
                ]):
                        tag.decompose()

                text = soup.get_text(separator=" ", strip=True)

                if text and len(text.strip()) > 200 :
                        cleaned = re.sub(r'\s+', ' ', text)
                        return cleaned[:5000]
                
                # strategy 3 -> fallback full page extraction

                soup = BeautifulSoup(html, "html.parser")

                for tag in soup([
                        "script",
                        "style",
                        "nav",
                        "footer",
                        "header",
                        "aside",
                        "form"
                ]):
                        tag.decompose()

                text = soup.get_text(separator=" ", strip=True)
                cleaned = re.sub(r'\s+', ' ', text)

                if cleaned :
                        return cleaned[:5000]
                
                return "Could not extract meaningful content from the page. "
        
        except requests.exceptions.Timeout:
                return "Request timed out while scraping the URL."
        
        except requests.exceptions.HTTPError as e:
                return f"HTTP error occured: {str(e)}"
        
        except Exception as e:
                return f"Could not scrape URL: {str(e)}"




In [5]:
scrape_url("https://www.district.in/movies/upcoming-movies")

"Stay ahead of every new release with District's complete upcoming movies calendar for 2026. From the biggest Bollywood blockbusters and Hollywood tentpoles to regional gems in Tamil, Telugu, Malayalam, Kannada, Bengali, Marathi, and Punjabi — find release dates, trailers, cast details, and pre-booking links all in one place. Whether you're planning a weekend movie night or tracking a long-awaited sequel, District helps you discover what's next in cinemas across Delhi, Mumbai, Bangalore, Hyderabad, Pune, Chennai, and Kolkata. Updated daily with new releases across every major language and city in India. Find upcoming releases in your preferred language — from mainstream Hindi and English to regional cinema in Tamil, Telugu, Malayalam, Kannada, Marathi, Bengali, and Punjabi. Discover the most anticipated Bollywood, South Indian, and regional films coming soon to theatres across India. Pre-book tickets in advance on District. Hindi, Telugu, Tamil, English, Malayalam, Kannada, Gujarati, P